In [1]:
%pip install selenium

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, WebDriverException
import time
from datetime import datetime
import re # For regular expressions to clean team names

In [1]:
from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd

# Define the URLs for the seasons you want to scrape
season_urls = {
    "2013-2014": "https://www.flashscore.es/futbol/inglaterra/premier-league-2013-2014/resultados/",
    "2014-2015": "https://www.flashscore.es/futbol/inglaterra/premier-league-2014-2015/resultados/",
    "2015-2016": "https://www.flashscore.es/futbol/inglaterra/premier-league-2015-2016/resultados/"
}

BATCH_SIZE = 75  # Restart browser after this many matches
all_commentary_data = []  # Store data from all seasons

for season, base_url in season_urls.items():
    print(f"\n--- Starting to scrape season: {season} ---")
    driver = webdriver.Chrome()
    driver.get(base_url)

    # Step 1: Click "show more" repeatedly to load all matches
    while True:
        try:
            show_more = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.XPATH, '//*[@id="live-table"]/div[1]/div/div/a'))
            )
            driver.execute_script("arguments[0].click();", show_more)
            time.sleep(2)
        except (NoSuchElementException, TimeoutException):
            print(f"No more 'show more' button found or timed out for {season}.")
            break

    # Step 2: Extract all match links and details
    current_season_match_details = []
    WebDriverWait(driver, 10).until(
        EC.presence_of_all_elements_located((By.XPATH, "//div[contains(@class, 'event__match--withRowLink')]"))
    )
    match_elements_rows = driver.find_elements(By.XPATH, "//div[contains(@class, 'event__match--withRowLink')]")
    for match_row_element in match_elements_rows:
        match_link = ""
        home_team = "N/A"
        away_team = "N/A"
        match_date = "N/A"

        try:
            link_element = match_row_element.find_element(By.XPATH, ".//a[@class='eventRowLink']")
            href = link_element.get_attribute("href")
            if href and "/#/resumen-del-partido" in href:
                match_link = href
        except NoSuchElementException:
            pass

        if not match_link:
            continue

        try:
            home_team_element = match_row_element.find_element(By.XPATH, ".//div[contains(@class, 'event__homeParticipant')]//*[contains(@class, 'wcl-name_3y6f5')]")
            home_team = home_team_element.text
        except NoSuchElementException:
            pass

        try:
            away_team_element = match_row_element.find_element(By.XPATH, ".//div[contains(@class, 'event__awayParticipant')]//*[contains(@class, 'wcl-name_3y6f5')]")
            away_team = away_team_element.text
        except NoSuchElementException:
            pass

        try:
            date_element = match_row_element.find_element(By.XPATH, ".//div[@class='event__time']")
            match_date = date_element.text
        except NoSuchElementException:
            pass

        current_season_match_details.append({
            "Season": season,
            "Match URL": match_link,
            "Home Team": home_team,
            "Away Team": away_team,
            "Match Date": match_date
        })

    print(f"Found {len(current_season_match_details)} match links with details for {season}.")
    driver.quit()  # Close after collecting links

    # Step 3: Scrape commentaries in batches
    for i in range(0, len(current_season_match_details), BATCH_SIZE):
        print(f"\n[INFO] Restarting browser... Batch starting from match {i+1}")
        driver = webdriver.Chrome()
        batch = current_season_match_details[i:i + BATCH_SIZE]

        for j, match_info in enumerate(batch):
            total_index = i + j
            url = match_info["Match URL"]
            home_team = match_info["Home Team"]
            away_team = match_info["Away Team"]
            match_date = match_info["Match Date"]
            current_season_name = match_info["Season"]

            print(f"Processing match {total_index+1}/{len(current_season_match_details)} ({current_season_name}): {home_team} vs {away_team} ({match_date})")

            if "#/resumen-del-partido/comentarios-en-directo" not in url:
                url_for_commentary = url.split('#')[0] + "#/resumen-del-partido/comentarios-en-directo/0"
            else:
                url_for_commentary = url

            current_match_commentary = {
                "Season": current_season_name,
                "Match URL": url,
                "Home Team": home_team,
                "Away Team": away_team,
                "Match Date": match_date,
                "Commentary": ""
            }

            try:
                driver.get(url_for_commentary)
                WebDriverWait(driver, 15).until(
                    EC.presence_of_element_located((By.XPATH, "//div[@class='section liveCommentary']"))
                )
                time.sleep(2)

                commentary_entries = driver.find_elements(By.XPATH, "//div[@class='section liveCommentary']//div[contains(@class, 'wcl-commentary_PDHM0')]")

                if commentary_entries:
                    full_commentary_text = "\n".join([entry.text for entry in commentary_entries if entry.text.strip()])
                    if full_commentary_text.strip():
                        current_match_commentary["Commentary"] = full_commentary_text
                    else:
                        print(f"Found containers but no text for {url}.")
                        current_match_commentary["Commentary"] = "Found containers but no detailed commentary text."
                else:
                    try:
                        no_commentary_message = driver.find_element(By.XPATH, "//div[@class='section liveCommentary']//div[contains(text(), 'No hay comentarios')]")
                        commentary_message = no_commentary_message.text.strip()
                        print(f"No commentary for {url}. Message: '{commentary_message}'")
                        current_match_commentary["Commentary"] = commentary_message
                    except NoSuchElementException:
                        print(f"No commentary elements or message found for {url}.")
                        current_match_commentary["Commentary"] = "No commentary available for this match."
            except NoSuchElementException as e:
                print(f"Error finding main container for {url_for_commentary}: {str(e)}. Skipping.")
                current_match_commentary["Commentary"] = f"Error: Main container not found ({str(e)})."
            except TimeoutException:
                print(f"Timeout while loading {url_for_commentary}. Skipping.")
                current_match_commentary["Commentary"] = "Timeout: Could not load commentary."
            except Exception as e:
                print(f"Unexpected error at {url_for_commentary}: {str(e)}. Skipping.")
                current_match_commentary["Commentary"] = f"Unexpected Error: {str(e)}"

            all_commentary_data.append(current_match_commentary)

        driver.quit()

# Step 4: Save the final data
df = pd.DataFrame(all_commentary_data)
output_filename = "flashscore_commentary_2013-2016.csv"
df.to_csv(output_filename, index=False)
print(f"\nDone: commentary for all seasons saved to {output_filename}")


--- Starting to scrape season: 2013-2014 ---
No more 'show more' button found or timed out for 2013-2014.
Found 380 match links with details for 2013-2014.

[INFO] Restarting browser... Batch starting from match 1
Processing match 1/380 (2013-2014): Cardiff vs Chelsea (11.05. 07:00)
Processing match 2/380 (2013-2014): Fulham vs Crystal Palace (11.05. 07:00)
Processing match 3/380 (2013-2014): Hull vs Everton (11.05. 07:00)
Processing match 4/380 (2013-2014): Liverpool vs Newcastle (11.05. 07:00)
Processing match 5/380 (2013-2014): Manchester City vs West Ham (11.05. 07:00)
Processing match 6/380 (2013-2014): Norwich vs Arsenal (11.05. 07:00)
Processing match 7/380 (2013-2014): Southampton vs Manchester Utd (11.05. 07:00)
Processing match 8/380 (2013-2014): Sunderland vs Swansea (11.05. 07:00)
Processing match 9/380 (2013-2014): Tottenham vs Aston Villa (11.05. 07:00)
Processing match 10/380 (2013-2014): West Brom vs Stoke (11.05. 07:00)
Processing match 11/380 (2013-2014): Manchester 

In [10]:
import pandas as pd

# Step 1: Load both CSV files
df1 = pd.read_csv('Livesum_Multilingual_processed_data.csv')
df2 = pd.read_csv('flashscore_commentary_2013-2016.csv')

# Step 2: Drop the unnecessary 'Multilingual Link' column from df1
df1 = df1.drop(columns=['Multilingual Link'])

# Step 3: Extract the season from 'Match Stats (Historical Data)' in df1 and store in new 'Season' column
df1['Match Stats (Historical Data)'] = df1['Match Stats (Historical Data)'].str.replace("Season: ", "")
df1['Season'] = df1['Match Stats (Historical Data)']

# Step 4: Reformat 'Season' in df2 from "20132014" to "2013-14"
df2['Season'] = df2['Season'].apply(lambda s: s[:4] + '-' + s[-2:])

# Step 5: Normalize the date format
# Convert df1['Date'] from "01/01/14" to "01.01"
df1['date'] = pd.to_datetime(df1['Date'], format='%d/%m/%y').dt.strftime('%d.%m')

# Extract "01.01" from df2['Match Date'] (format: "01.01 10:30")
df2['date'] = df2['Match Date'].str.extract(r'(\d{2}\.\d{2})')

# Step 6: Create a merge key in both DataFrames
df1['merge_key'] = df1['Season'] + '_' + df1['HomeTeam'] + '_' + df1['AwayTeam'] + '_' + df1['date']
df2['merge_key'] = df2['Season'] + '_' + df2['Home Team'] + '_' + df2['Away Team'] + '_' + df2['date']

# Step 7: Merge both DataFrames on the 'merge_key' using a left join
merged_df = pd.merge(df1, df2, on='merge_key', how='left')

# Step 8: Drop duplicate or unnecessary columns after merge
merged_df.drop(columns=['Season_x', 'date_x', 'merge_key', 'Season_y', 'date_y', 'Home Team', 'Away Team'], inplace=True)

# Step 9: Extract time ("10:30") from 'Match Date' in df2
merged_df['Time'] = merged_df['Match Date'].str.extract(r'(\d{2}:\d{2})')

# Step 10: Combine original 'Date' column from df1 with the extracted time
merged_df['Combined'] = merged_df['Date'] + ' ' + merged_df['Time']

# Step 11: Drop the old 'Date', 'Match Date', and 'Time' columns
merged_df.drop(columns=['Match Date', 'Date', 'Time'], inplace=True)

# Step 12: Rename 'Combined' to 'Date' and reposition it after the 'Match Stats (Historical Data)' column
merged_df.rename(columns={'Combined': 'Date'}, inplace=True)
date_col = merged_df.pop('Date')
merged_df.insert(merged_df.columns.get_loc('Match Stats (Historical Data)') + 1, 'Date', date_col)

# Step 13: Rename 'Match Stats (Historical Data)' to 'Season'
merged_df.rename(columns={'Match Stats (Historical Data)': 'Season'}, inplace=True)

# Step 14: Output the final DataFrame and save to CSV
print("Unified Date column added successfully!")
print("\nFirst 5 rows of the merged DataFrame:")
display(merged_df.head())
print(f"\nShape of the merged DataFrame: {merged_df.shape}")
merged_df.to_csv('Flashscore_Final.csv', index=False)

Unified Date column added successfully!

First 5 rows of the merged DataFrame:


,Sample #,ID in Training Set,Season,Date,HomeTeam,AwayTeam,Match URL,Commentary
0,2,25513267,2013-14,01/01/14 10:30,Man United,Tottenham,https://www.flashscore.es/partido/futbol/0GvFE...,90+6'\nFinal del partido.\n90+4'\nShinji Kagaw...
1,3,25513268,2013-14,01/01/14 08:00,West Brom,Newcastle,https://www.flashscore.es/partido/futbol/SSF9z...,90+5'\nLee Mason mira su reloj y pita el final...
2,4,25513270,2013-14,01/01/14 08:00,Liverpool,Hull,https://www.flashscore.es/partido/futbol/IikAF...,90+3'\nFinal del partido.\n90+2'\nPhilippe Cou...
3,5,25513272,2013-14,01/01/14 08:00,Stoke,Everton,https://www.flashscore.es/partido/futbol/nXxNC...,90+5'\nEl colegiado mira su reloj y señala el ...
4,6,25513273,2013-14,01/01/14 05:45,Swansea,Man City,https://www.flashscore.es/partido/futbol/O80nl...,90+4'\nNo hay tiempo para más. El árbitro seña...



Shape of the merged DataFrame: (400, 8)
